In [ ]:
!hostname

: 

In [ ]:

import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"


from photometry.models.baselines import LommelSeeligerModel
from photometry.fitting.least_sq import LeastSquaresFitter
from photometry.core.types import GeometryBatch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path
import dataclasses
!pip install rich
from rich import print
import os
print(os.getcwd())

: 

In [ ]:
# Locate aggregated phase-curve CSVs

project_root = Path.cwd().resolve()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent
data_root = project_root / "data" / "05_aggregated"
csv_files = sorted(data_root.glob("*_phase_curve.csv"))
phase_dfs = {}
for path in csv_files:
    phase = path.stem.replace("_phase_curve", "").lower()
    phase_dfs[phase] = pd.read_csv(path)

# Ensure phase DataFrames exist
df_rc = phase_dfs.get("rc", pd.DataFrame()).copy()
df_survey = phase_dfs.get("survey", pd.DataFrame()).copy()
df_hamo = phase_dfs.get("hamo", pd.DataFrame()).copy()
df_lamo = phase_dfs.get("lamo", pd.DataFrame()).copy()

# Apply incidence quality filter: mean_incidence <= 60.0 degrees
def apply_incidence_filter(df):
    if df is None or df.empty:
        return df
    return df[(df["mean_incidence"] <= 60.0) & (df["median_iof"] > 0.01)].dropna().copy()

rc_f = apply_incidence_filter(df_rc)
survey_f = apply_incidence_filter(df_survey)
hamo_f = apply_incidence_filter(df_hamo)
lamo_f = apply_incidence_filter(df_lamo)


print("RC, Survey, HAMO, and LAMO phase-curve data (first 5 rows each):\n")
display(df_rc.head())
display(df_survey.head())    
display(df_hamo.head())
display(df_lamo.head())

print("\nAfter applying incidence quality filter (mean_incidence <= 60.0 degrees) and (median_iof > 0.01):\n")

display(rc_f.head())
display(survey_f.head())    
display(hamo_f.head())
display(lamo_f.head())


# Extract n_pixels array from each phase DataFrame
    
rc_n_pixels = rc_f["n_pixels"].to_numpy(dtype=float) if "n_pixels" in rc_f.columns else None
survey_n_pixels = survey_f["n_pixels"].to_numpy(dtype=float) if "n_pixels" in survey_f.columns else None
hamo_n_pixels = hamo_f["n_pixels"].to_numpy(dtype=float) if "n_pixels" in hamo_f.columns else None
lamo_n_pixels = lamo_f["n_pixels"].to_numpy(dtype=float) if "n_pixels" in lamo_f.columns else None


n_pixels_arrays = [rc_n_pixels, survey_n_pixels, hamo_n_pixels, lamo_n_pixels]
print("n_pixels_arrays:", n_pixels_arrays)

In [ ]:

phases = [("RC", rc_f), ("Survey", survey_f), ("HAMO", hamo_f), ("LAMO", lamo_f)]
fitter = LeastSquaresFitter()
published_w = 0.38  # reference from Schröder et al. 2013

results = {}
for phase_name, df_phase in phases:
    if df_phase is None or df_phase.empty:
        print(f"{phase_name}: no data after 60° incidence filter")
        continue

    incidence_rad = np.deg2rad(df_phase["mean_incidence"].to_numpy(dtype=float))
    emission_rad = np.deg2rad(df_phase["mean_emission"].to_numpy(dtype=float))
    phase_rad = np.deg2rad(df_phase["mean_phase"].to_numpy(dtype=float))

    geom_batch = GeometryBatch(
        incidence=incidence_rad,
        emission=emission_rad,
        phase=phase_rad,
    )

    observed = df_phase["median_iof"].to_numpy(dtype=float) # observed I/F values
    weights = df_phase["n_pixels"].to_numpy(dtype=float) if "n_pixels" in df_phase.columns else None # weights based on n_pixels if available, else None for unweighted fit 
    
    model_instance = LommelSeeligerModel()
    fit_result = fitter.fit(model_instance, geom_batch, observed, weights=None) # Use unweighted fit for now; can switch to weights=weights if desired

    w_fit = fit_result.fitted_parameters.get("w")
    w_err = fit_result.metadata.get("parameter_errors", {}).get("w", None)
    chi_sq = fit_result.metadata.get("reduced_chi_square", None)

    results[phase_name] = {
        "w": float(w_fit),
        "w_err": float(w_err) if w_err is not None else None,
        "n_bins": int(len(df_phase)),
        "reduced_chi_square": float(chi_sq) if chi_sq is not None else None,
    }



    print(f"{phase_name}: fitted w = {w_fit:.6f} ± {w_err:.6f}; bins after filter = {len(df_phase)}")


In [ ]:
results_df = pd.DataFrame.from_dict(results, orient="index")
print("\nSummary of Lommel-Seeliger fit results for each phase:")
display(results_df)

In [ ]:
# Visualization: Disk-corrected I/F vs Phase for combined RC+Survey

combined_df = pd.concat([rc_f.assign(phase_name="RC"), survey_f.assign(phase_name="Survey")], ignore_index=True)
if combined_df.empty:
    print("No RC+Survey rows after filtering; skipping plot.")
else:
    inc_rad = np.deg2rad(combined_df["mean_incidence"].to_numpy(dtype=float))
    emi_rad = np.deg2rad(combined_df["mean_emission"].to_numpy(dtype=float))
    mu0 = np.cos(inc_rad)
    mu = np.cos(emi_rad)
    disk_func = mu0 / (mu0 + mu + 1e-10)
    valid = disk_func > 1e-12
    plot_df = combined_df[valid].copy()
    plot_df["disk_corrected_iof"] = plot_df["median_iof"].to_numpy(dtype=float) / disk_func[valid]
    fig, ax = plt.subplots(figsize=(10, 6))
    for label, group in plot_df.groupby("phase_name"):
        ax.scatter(group["phase_bin_deg"].to_numpy(dtype=float), group["disk_corrected_iof"].to_numpy(dtype=float), label=label, s=30, alpha=0.8)
    ax.set_xlabel("Phase Angle (deg)")
    ax.set_ylabel("Disk-Corrected I/F")
    ax.set_title("RC + Survey Disk-Corrected I/F vs Phase (mean_incidence ≤ 60°)")
    ax.legend()
    ax.grid(True, alpha=0.25)
    plt.show()

In [ ]:
# Calculate Lommel-Seeliger Disk-Corrected I/F for all four phases
import numpy as np
import matplotlib.pyplot as plt

# Prepare data for all phases
phases_data = [
    ("RC", rc_f),
    ("Survey", survey_f),
    ("HAMO", hamo_f),
    ("LAMO", lamo_f),
]

plot_dfs = []
for phase_name, df in phases_data:
    if df is None or df.empty:
        continue
    
    # Convert angles to radians
    incidence_rad = np.deg2rad(df["mean_incidence"].to_numpy(dtype=float))
    emission_rad = np.deg2rad(df["mean_emission"].to_numpy(dtype=float))
    
    # Calculate Lommel-Seeliger Disk Function: D = cos(i) / (cos(i) + cos(e))
    cos_i = np.cos(incidence_rad)
    cos_e = np.cos(emission_rad)
    disk_func = cos_i / (cos_i + cos_e + 1e-10)  # small epsilon to avoid division by zero
    
    # Calculate disk-corrected I/F
    iof = df["median_iof"].to_numpy(dtype=float)
    corrected_iof = iof / disk_func
    
    # Create DataFrame for this phase
    phase_plot_df = df.copy()
    phase_plot_df["corrected_iof"] = corrected_iof
    phase_plot_df["phase_name"] = phase_name
    plot_dfs.append(phase_plot_df)

# Combine all phases
combined_plot_df = pd.concat(plot_dfs, ignore_index=True)

# Create scatter plot
fig, ax = plt.subplots(figsize=(12, 7))

colors = {"RC": "#1f77b4", "Survey": "#ff7f0e", "HAMO": "#2ca02c", "LAMO": "#d62728"}
for phase_name in ["RC", "Survey", "HAMO", "LAMO"]:
    phase_data = combined_plot_df[combined_plot_df["phase_name"] == phase_name]
    if not phase_data.empty:
        ax.scatter(
            phase_data["mean_phase"].to_numpy(dtype=float),
            phase_data["corrected_iof"].to_numpy(dtype=float),
            label=phase_name,
            color=colors[phase_name],
            s=40,
            alpha=0.6,
            edgecolors="none"
        )

ax.set_xlabel("Phase Angle (degrees)", fontsize=12)
ax.set_ylabel("Disk-Corrected I/F", fontsize=12)
ax.set_title("Lommel-Seeliger Disk-Corrected I/F vs Phase Angle\n(mean_incidence ≤ 60°)", fontsize=13)
ax.set_xlim(0, 100)
ax.grid(True, alpha=0.3, linestyle="--")
ax.legend(loc="best", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
display(df_rc[df_rc['median_iof'] <= 0.01].count())
display(df_rc[df_rc['median_iof'] <= 0.01]

In [ ]:
display(df_survey[df_survey['median_iof'] <= 0.01].count())

In [ ]:
display(df_hamo[df_hamo['median_iof'] <= 0.01].count())

In [ ]:
display(df_lamo[df_lamo['median_iof'] <= 0.01].count())

In [ ]:
# Median I/F <= 0.01 and n_pixels only, for all phases
rc_small = df_rc.loc[df_rc["median_iof"] <= 0.01, ["median_iof", "n_pixels"]].copy()
survey_small = df_survey.loc[df_survey["median_iof"] <= 0.01, ["median_iof", "n_pixels"]].copy()
hamo_small = df_hamo.loc[df_hamo["median_iof"] <= 0.01, ["median_iof", "n_pixels"]].copy()
lamo_small = df_lamo.loc[df_lamo["median_iof"] <= 0.01, ["median_iof", "n_pixels"]].copy()

display(rc_small)
display(survey_small)
display(hamo_small)
display(lamo_small)

In [ ]:
rc_n_pixels = df_rc.loc[df_rc["median_iof"] <= 0.01, "n_pixels"].to_numpy(dtype=float)
survey_n_pixels = df_survey.loc[df_survey["median_iof"] <= 0.01, "n_pixels"].to_numpy(dtype=float)
hamo_n_pixels = df_hamo.loc[df_hamo["median_iof"] <= 0.01, "n_pixels"].to_numpy(dtype=float)
lamo_n_pixels = df_lamo.loc[df_lamo["median_iof"] <= 0.01, "n_pixels"].to_numpy(dtype=float)

print('rc_n_pixels:', rc_n_pixels)
print('survey_n_pixels:', survey_n_pixels)
print('hamo_n_pixels:', hamo_n_pixels)
print('lamo_n_pixels:', lamo_n_pixels)